In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from datetime import datetime
import pytz
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('Play Store Data.csv')
print(f"Raw shape: {df.shape}")
df.head()

In [ ]:
# drop rows missing the columns we'll filter on
df.dropna(subset=['Category', 'Installs', 'Size', 'Android Ver', 'Type', 'Content Rating'], inplace=True)

# --- Installs: strip commas and plus signs, then cast ---
df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.strip()
)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df.dropna(subset=['Installs'], inplace=True)

# --- Price: remove dollar signs, fill NaN with 0 ---
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0)

# Revenue = Price x Installs
df['Revenue'] = df['Price'] * df['Installs']

# --- Size: only rows with 'M' suffix, then strip and convert ---
df = df[df['Size'].astype(str).str.endswith('M')]
df['Size_num'] = (
    df['Size']
    .astype(str)
    .str.replace('M', '', regex=False)
    .str.strip()
)
df['Size_num'] = pd.to_numeric(df['Size_num'], errors='coerce')
df.dropna(subset=['Size_num'], inplace=True)

# --- Android Version: extract first numeric version like '4.1' ---
df['Android_num'] = (
    df['Android Ver']
    .astype(str)
    .str.extract(r'(\d+\.\d+)', expand=False)
)
df['Android_num'] = pd.to_numeric(df['Android_num'], errors='coerce')
df.dropna(subset=['Android_num'], inplace=True)

print(f"After cleaning: {df.shape}")

In [ ]:
# app name length — count all characters including spaces and special characters
df['name_len'] = df['App'].astype(str).apply(len)

df_filtered = df[
    (df['Installs'] >= 10000) &
    (df['Revenue'] >= 10000) &
    (df['Android_num'] > 4.0) &
    (df['Size_num'] > 15) &
    (df['Content Rating'] == 'Everyone') &
    (df['name_len'] <= 30)
].copy()

# keep only valid Type entries
df_filtered = df_filtered[df_filtered['Type'].isin(['Free', 'Paid'])]

print(f"After all filters: {df_filtered.shape}")
print("Type counts:\n", df_filtered['Type'].value_counts())

In [ ]:
# top 3 categories based on total installs after filtering
cat_totals = (
    df_filtered
    .groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
)

top3_cats = cat_totals.head(3).index.tolist()
print("Top 3 Categories:", top3_cats)
print(cat_totals.head(3))

In [ ]:
plot_df = df_filtered[df_filtered['Category'].isin(top3_cats)].copy()

# average installs and average revenue grouped by category + type
agg = (
    plot_df
    .groupby(['Category', 'Type'])
    .agg(
        avg_installs=('Installs', 'mean'),
        avg_revenue=('Revenue', 'mean')
    )
    .reset_index()
)
print(agg)

In [ ]:
# visibility window: 1:00 PM to 2:00 PM IST
ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)
decimal_time = now_ist.hour + now_ist.minute / 60

in_window = (13.0 <= decimal_time < 14.0)

print(f"Current IST: {now_ist.strftime('%I:%M %p')}")
print(f"Chart visible: {in_window}")

In [ ]:
chart_html = ''

if not in_window:
    # outside the display window - chart container will be hidden entirely in dashboard
    print(f"Not in display window. Current time: {now_ist.strftime('%I:%M %p')} IST")

elif agg.empty:
    chart_html = "<p style='color:#ef4444;text-align:center;padding:40px;'>No data after filtering.</p>"

else:
    x_labels = []
    installs_vals = []
    revenue_vals = []
    bar_colors = []

    for cat in top3_cats:
        for app_type in ['Free', 'Paid']:
            row = agg[(agg['Category'] == cat) & (agg['Type'] == app_type)]
            x_labels.append(f"{cat.replace('_', ' ').title()}<br>{app_type}")
            if not row.empty:
                installs_vals.append(round(row['avg_installs'].values[0], 0))
                revenue_vals.append(round(row['avg_revenue'].values[0], 2))
            else:
                installs_vals.append(0)
                revenue_vals.append(0)
            bar_colors.append('#4ade80' if app_type == 'Free' else '#f97316')

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    # bars for avg installs - primary y-axis
    fig.add_trace(
        go.Bar(
            x=x_labels,
            y=installs_vals,
            name='Avg Installs',
            marker_color=bar_colors,
            opacity=0.82,
            hovertemplate='<b>%{x}</b><br>Avg Installs: %{y:,.0f}<extra></extra>'
        ),
        secondary_y=False
    )

    # line for avg revenue - secondary y-axis
    fig.add_trace(
        go.Scatter(
            x=x_labels,
            y=revenue_vals,
            name='Avg Revenue ($)',
            mode='lines+markers',
            line=dict(color='#a78bfa', width=2.5),
            marker=dict(size=9, color='#a78bfa', symbol='circle'),
            hovertemplate='<b>%{x}</b><br>Avg Revenue: $%{y:,.2f}<extra></extra>'
        ),
        secondary_y=True
    )

    fig.update_layout(
        title=dict(
            text='Avg Installs vs Avg Revenue — Free vs Paid Apps across Top 3 Categories',
            font=dict(size=16, color='#e5e7eb'),
            x=0.5,
            xanchor='center'
        ),
        paper_bgcolor='#1e1e2e',
        plot_bgcolor='#1e1e2e',
        font=dict(color='#e5e7eb', size=12),
        legend=dict(
            bgcolor='#2a2a3e',
            bordercolor='#444466',
            borderwidth=1,
            font=dict(size=12)
        ),
        xaxis=dict(
            title=dict(text='App Type by Category', font=dict(size=12)),
            tickfont=dict(size=10),
            gridcolor='#2a2a3e',
            linecolor='#444466'
        ),
        margin=dict(l=70, r=70, t=80, b=90),
        height=530,
        width=1050,
        bargap=0.3
    )

    fig.update_yaxes(
        title_text='Average Installs',
        gridcolor='#2a2a3e',
        linecolor='#444466',
        secondary_y=False
    )
    fig.update_yaxes(
        title_text='Average Revenue ($)',
        gridcolor='#2a2a3e',
        linecolor='#444466',
        secondary_y=True
    )

    chart_html = pio.to_html(fig, full_html=False, include_plotlyjs='cdn')
    fig.write_html('dual_axis_chart.html', full_html=False, include_plotlyjs='cdn')
    fig.show()
    print("Chart saved: dual_axis_chart.html")

In [ ]:
# build summary table rows
summary_rows = ''
for rank, cat in enumerate(top3_cats, 1):
    total = int(cat_totals[cat])
    summary_rows += (
        f'<tr><td>{rank}</td>'
        f'<td>{cat.replace("_", " ").title()}</td>'
        f'<td>{total:,}</td></tr>\n'
    )

# chart container is hidden entirely when chart_html is empty (outside window)
container_style = 'display:none;' if not chart_html else ''
time_note = (
    '<p class="time-note">&#9200; Dual-axis chart visible only between 1:00 PM and 2:00 PM IST.</p>'
    if not in_window else ''
)

dashboard = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Play Store Dual-Axis Dashboard</title>
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ background: #13131f; color: #e5e7eb; font-family: "Segoe UI", system-ui, sans-serif; min-height: 100vh; padding: 28px 22px; }}
  header {{ text-align: center; margin-bottom: 30px; }}
  header h1 {{ font-size: 26px; font-weight: 700; color: #f0f0ff; letter-spacing: 0.4px; }}
  header p {{ font-size: 13px; color: #9ca3af; margin-top: 7px; }}
  .section-title {{ font-size: 12px; font-weight: 600; color: #9ca3af; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 12px; }}
  .chart-box {{ background: #1e1e2e; border: 1px solid #2a2a4a; border-radius: 12px; padding: 20px; margin-bottom: 28px; overflow-x: auto; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  thead tr {{ background: #2a2a3e; }}
  thead th {{ padding: 11px 14px; text-align: left; color: #9ca3af; font-weight: 600; font-size: 11px; text-transform: uppercase; letter-spacing: 0.5px; }}
  tbody tr {{ border-bottom: 1px solid #2a2a3e; }}
  tbody tr:hover {{ background: #25253a; }}
  tbody td {{ padding: 10px 14px; color: #e5e7eb; }}
  .time-note {{ font-size: 12px; color: #6b7280; text-align: center; margin-top: -14px; margin-bottom: 20px; }}
</style>
</head>
<body>
<header>
  <h1>Play Store Analytics Dashboard</h1>
  <p>Free vs Paid — Top 3 Categories by Total Installs</p>
</header>

<div class="chart-box">
  <div class="section-title">Top 3 Categories by Total Installs</div>
  <table>
    <thead><tr><th>#</th><th>Category</th><th>Total Installs</th></tr></thead>
    <tbody>{summary_rows}</tbody>
  </table>
</div>

{time_note}

<div class="chart-box" style="{container_style}">
  <div class="section-title">Avg Installs vs Avg Revenue (1:00 PM to 2:00 PM IST only)</div>
  {chart_html}
</div>

</body>
</html>'''

with open('dual_axis_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard)

print("Dashboard saved: dual_axis_dashboard.html")